# **🏢 Amazon ML Challenge 2026 — Business Entity Resolution**

This project focuses on matching noisy business entity records across independent data sources without shared unique IDs.

## **🧩 02. Candidate Generation (Blocking) Pipeline**

This notebook handles the scalable generation of candidate pairs using DuckDB. By implementing targeted blocking strategies, we reduce trillions of potential cross-comparisons down to a manageable, high-recall dataset for downstream machine learning.

---

### **1. Problem Definition**

Comparing 2.2 million reference records against 10 million query records requires over 22 trillion direct comparisons. Blocking acts as a high-recall filter, grouping records into overlapping "buckets" based on shared traits (tokens, sorted words, country).

Our absolute priority in this phase is **Recall**: a true match dropped during blocking can never be recovered by the classifier later.

> **Key Constraint:** `clean_address` is missing in ~3.3% of Source 2 and Source 3 records. We cannot rely strictly on address intersections, or we permanently lose thousands of true matches.

---

### **2. Data Inputs & Architecture**

This pipeline ingests the optimized Parquet files generated in the Phase 1 Data Cleaning notebook, alongside the raw Ground Truth mapping.

**Input Datasets:**

* `clean_source1.parquet`: Ground reference anchor
* `clean_source2.parquet`: Noisy query source 1
* `clean_source3.parquet`: Noisy query source 2
* `train_ground_truth.tsv`: Used **only** at the final step to assign `1` (Match) or `0` (Non-Match) labels to our generated candidate pairs.

**Available Schema:**

| **Column** | **Type** | **Description** |
| --- | --- | --- |
| `entity_id` | String | Unique record ID (`S1-`, `S2-`, `S3-`) |
| `country` | String | Hard partition key (`US`, `India`, `France`) |
| `clean_name` | String | Standardized business name (lowercased, exact legal suffixes stripped) |
| `clean_address` | String | Standardized address (contains nulls) |

---

### **3. Pipeline Execution Steps**

We generate candidate pairs through a multi-net approach, executing independently and combining via union. A pair only needs to survive *one* of the following techniques to be passed to the feature engineering phase.

1. **Hard Country Partitioning:** Strictly isolate comparisons by region. Never compare records across different countries.
2. **Primary Net (Sorted Token Key):** Split `clean_name` into distinct words, sort them alphabetically, and rejoin (e.g., *moncada learning center* $\to$ *center learning moncada*) to catch out-of-order words.
3. **Secondary Net (Exploded Per-Token):** Explode `clean_name` so records are bucketed by every individual word they contain, ensuring recall even if a word is dropped or added.
4. **Tertiary Net (Address Tokens):** For non-null addresses, group by extracted city, state, or zip tokens to catch matches where the business name is heavily mangled.
5. **Union & Deduplication:** Combine all candidate pairs across the three nets, keeping a record of which technique(s) (`matched_via`) successfully generated the match.
6. **Ground Truth Labeling:** Join the final deduplicated candidate pairs against the exploded `train_ground_truth.tsv` to assign binary labels for the classifier.

### **1. Loading the Clean Dataset Directly from Storage (Out-of-Core)**
Instead of loading 12 million rows into pandas and crashing our RAM, we map DuckDB directly to our Parquet files. DuckDB acts as a high-performance query engine, streaming data from the SSD in chunks rather than stuffing it all into memory. 

By setting `PRAGMA memory_limit`, we safely cap RAM usage while keeping the CPU fully utilized.

In [1]:
import duckdb
import time
from pathlib import Path

# Adjust path if your clean_data folder is located elsewhere
DATA_DIR = Path("../data/clean_data")

# Initialize DuckDB connection
con = duckdb.connect(database=':memory:')

# Hard-cap DuckDB's memory usage to protect your system from crashing
con.execute("PRAGMA memory_limit='10GB'")

# Create virtual views pointing directly to the Parquet files on disk
# This takes 0 seconds and uses 0 MB of RAM!
print("Linking datasets to DuckDB...")
con.execute(f"CREATE VIEW s1 AS SELECT * FROM read_parquet('{DATA_DIR / 'clean_source1.parquet'}')")
con.execute(f"CREATE VIEW s2 AS SELECT * FROM read_parquet('{DATA_DIR / 'clean_source2.parquet'}')")
con.execute(f"CREATE VIEW s3 AS SELECT * FROM read_parquet('{DATA_DIR / 'clean_source3.parquet'}')")

# Quick test to prove it works instantly
print("S2 Row Count:")
print(con.execute("SELECT COUNT(*) FROM s2").fetchone()[0])

Linking datasets to DuckDB...
S2 Row Count:
5034616


### **2a. Primary Net: Sorted Token Blocking**
A strict exact match fails if words are swapped (e.g., "moncada learning center" vs "learning center moncada"). To fix this, we:
1. Split `clean_name` into an array of individual words.
2. Sort the array alphabetically.
3. Cast it back to a string to use as a blocking key.

We also strictly enforce a `country` match. A business does not silently change countries, so partitioning by country instantly halves our compute time and eliminates millions of impossible pairs.

In [2]:
print("Generating candidate pairs for Step 2a (Sorted Token Match)...")
start = time.time()

# We execute this for Source 1 -> Source 2 first
con.execute("""
    CREATE TABLE pairs_2a_s2 AS
    WITH s1_keys AS (
        SELECT 
            entity_id AS s1_id, 
            country,
            list_sort(string_split(clean_name, ' '))::VARCHAR AS sorted_key
        FROM s1
        WHERE clean_name IS NOT NULL AND clean_name != ''
    ),
    s2_keys AS (
        SELECT 
            entity_id AS s2_id, 
            country,
            list_sort(string_split(clean_name, ' '))::VARCHAR AS sorted_key
        FROM s2
        WHERE clean_name IS NOT NULL AND clean_name != ''
    )
    SELECT 
        s1.s1_id AS source1_entity_id, 
        s2.s2_id AS candidate_entity_id,
        'S2' AS source,
        '2a_sorted_tokens' AS matched_via
    FROM s1_keys s1
    INNER JOIN s2_keys s2
        ON s1.country = s2.country 
        AND s1.sorted_key = s2.sorted_key;
""")

# Count the generated pairs
count_2a_s2 = con.execute("SELECT COUNT(*) FROM pairs_2a_s2").fetchone()[0]
print(f"Step 2a completed in {time.time() - start:.2f} seconds.")
print(f"Generated {count_2a_s2:,} candidate pairs from Source 2.")

Generating candidate pairs for Step 2a (Sorted Token Match)...
Step 2a completed in 10.60 seconds.
Generated 39,914,895 candidate pairs from Source 2.


### **2a (Part 2): Sorted Token Blocking for Source 3**
We applied the sorted token key logic to Source 2. Now we apply the exact same process to Source 3, joining `clean_source1` against `clean_source3` strictly within the same country boundaries.

In [3]:
print("Generating candidate pairs for Step 2a (Source 1 -> Source 3)...")
start = time.time()

con.execute("""
    CREATE TABLE pairs_2a_s3 AS
    WITH s1_keys AS (
        SELECT 
            entity_id AS s1_id, 
            country,
            list_sort(string_split(clean_name, ' '))::VARCHAR AS sorted_key
        FROM s1
        WHERE clean_name IS NOT NULL AND clean_name != ''
    ),
    s3_keys AS (
        SELECT 
            entity_id AS s3_id, 
            country,
            list_sort(string_split(clean_name, ' '))::VARCHAR AS sorted_key
        FROM s3
        WHERE clean_name IS NOT NULL AND clean_name != ''
    )
    SELECT 
        s1.s1_id AS source1_entity_id, 
        s3.s3_id AS candidate_entity_id,
        'S3' AS source,
        '2a_sorted_tokens' AS matched_via
    FROM s1_keys s1
    INNER JOIN s3_keys s3
        ON s1.country = s3.country 
        AND s1.sorted_key = s3.sorted_key;
""")

# Count the generated pairs
count_2a_s3 = con.execute("SELECT COUNT(*) FROM pairs_2a_s3").fetchone()[0]
print(f"Step 2a (S3) completed in {time.time() - start:.2f} seconds.")
print(f"Generated {count_2a_s3:,} candidate pairs from Source 3.")

Generating candidate pairs for Step 2a (Source 1 -> Source 3)...
Step 2a (S3) completed in 11.94 seconds.
Generated 44,390,988 candidate pairs from Source 3.


### **2a (Part 3): Previewing the Generated Pairs**
To verify our sorted token logic worked, we join a few generated candidate pairs back to their original datasets to view the `clean_name` strings side-by-side. We should expect to see identical names, or names where the exact same words appear but in a different order.

In [4]:
import pandas as pd
from IPython.display import display

print("Previewing Step 2a matches (Source 1 -> Source 2)...")
s2_preview = con.execute("""
    SELECT 
        p.source1_entity_id, 
        s1.clean_name AS s1_name, 
        p.candidate_entity_id AS s2_entity_id, 
        s2.clean_name AS s2_name
    FROM pairs_2a_s2 p
    JOIN s1 ON p.source1_entity_id = s1.entity_id
    JOIN s2 ON p.candidate_entity_id = s2.entity_id
    LIMIT 5
""").df()
display(s2_preview)

print("\nPreviewing Step 2a matches (Source 1 -> Source 3)...")
s3_preview = con.execute("""
    SELECT 
        p.source1_entity_id, 
        s1.clean_name AS s1_name, 
        p.candidate_entity_id AS s3_entity_id, 
        s3.clean_name AS s3_name
    FROM pairs_2a_s3 p
    JOIN s1 ON p.source1_entity_id = s1.entity_id
    JOIN s3 ON p.candidate_entity_id = s3.entity_id
    LIMIT 5
""").df()
display(s3_preview)

Previewing Step 2a matches (Source 1 -> Source 2)...


,source1_entity_id,s1_name,s2_entity_id,s2_name
0,S1-283174598,dermatology care associates,S2-895674743,dermatology care associates
1,S1-428253953,big it,S2-554929444,big it
2,S1-396728893,blue ice cream,S2-606954109,blue ice cream
3,S1-846196148,rev it up tavern,S2-96015099,rev it up tavern
4,S1-64404097,uptown bakery,S2-996921067,uptown bakery



Previewing Step 2a matches (Source 1 -> Source 3)...


,source1_entity_id,s1_name,s3_entity_id,s3_name
0,S1-271671813,united partners,S3-112280953,united partners
1,S1-838133999,sequoia,S3-950521712,sequoia
2,S1-79336003,valley academy,S3-209205020,valley academy
3,S1-345420230,human rights league,S3-758939079,human league rights
4,S1-660314668,regional fellowship,S3-611931975,regional fellowship


### **2a (Part 4): Country Verification & Distribution**
We add the `country` column to our preview and check the pair counts for US vs India to confirm that cross-country matches were strictly prevented.

In [5]:
# 1. Preview with country column included
print("Previewing Step 2a matches with Country column...")
preview_with_country = con.execute("""
    SELECT 
        s1.country,
        p.source1_entity_id, 
        s1.clean_name AS s1_name, 
        p.candidate_entity_id AS s3_entity_id, 
        s3.clean_name AS s3_name
    FROM pairs_2a_s3 p
    JOIN s1 ON p.source1_entity_id = s1.entity_id
    JOIN s3 ON p.candidate_entity_id = s3.entity_id
    LIMIT 5
""").df()
display(preview_with_country)

# 2. Check exact pair breakdown per country
print("\nSource 2 Pair Distribution by Country:")
display(con.execute("""
    SELECT s1.country, COUNT(*) AS total_pairs
    FROM pairs_2a_s2 p
    JOIN s1 ON p.source1_entity_id = s1.entity_id
    GROUP BY s1.country
""").df())

print("\nSource 3 Pair Distribution by Country:")
display(con.execute("""
    SELECT s1.country, COUNT(*) AS total_pairs
    FROM pairs_2a_s3 p
    JOIN s1 ON p.source1_entity_id = s1.entity_id
    GROUP BY s1.country
""").df())

Previewing Step 2a matches with Country column...


,country,source1_entity_id,s1_name,s3_entity_id,s3_name
0,US,S1-873281511,physical therapy specialists,S3-16785783,physical therapy specialists
1,US,S1-966015743,womens health specialists,S3-906788611,womens health specialists
2,US,S1-468065365,the dent seafood,S3-512685247,the dent seafood
3,US,S1-461644558,urgent care physicians,S3-361023032,urgent care physicians
4,US,S1-309083746,slate,S3-693812493,slate



Source 2 Pair Distribution by Country:


,country,total_pairs
0,India,7566292
1,US,32348603



Source 3 Pair Distribution by Country:


,country,total_pairs
0,US,32383352
1,India,12007636


### **2b. Secondary Net: Per-Token Word Explode Blocking**
To handle cases where a word was added or dropped across sources (e.g., *"EFS Print Ventures"* vs *"EFS Print Ventures Group"*), we explode `clean_name` into individual word tokens.

**Guardrails Enforced:**
- Minimum token length $\ge 3$ characters.
- Token frequency capped at $\le 300$ occurrences per country to eliminate generic words (`"care"`, `"center"`, `"group"`).
- Strict country alignment (`country` match).

In [10]:
# Python Code Cell: Step 2b for Source 2

print("Generating candidate pairs for Step 2b (Per-Token Explode - Source 2)...")
start = time.time()

con.execute("""
    CREATE TABLE pairs_2b_s2 AS
    WITH s1_tokens AS (
        SELECT 
            entity_id AS s1_id,
            country,
            UNNEST(string_split(clean_name, ' ')) AS token
        FROM s1
        WHERE clean_name IS NOT NULL AND clean_name != ''
    ),
    s2_tokens AS (
        SELECT 
            entity_id AS s2_id,
            country,
            UNNEST(string_split(clean_name, ' ')) AS token
        FROM s2
        WHERE clean_name IS NOT NULL AND clean_name != ''
    ),
    -- Guardrail: Identify high-frequency tokens per country in S1 to drop
    valid_s1_tokens AS (
        SELECT 
            s1_id,
            country,
            token
        FROM (
            SELECT 
                s1_id,
                country,
                token,
                COUNT(*) OVER (PARTITION BY country, token) AS token_freq
            FROM s1_tokens
            WHERE LENGTH(token) >= 3
        )
        WHERE token_freq <= 300
    )
    SELECT DISTINCT
        s1.s1_id AS source1_entity_id,
        s2.s2_id AS candidate_entity_id,
        'S2' AS source,
        '2b_token_explode' AS matched_via
    FROM valid_s1_tokens s1
    INNER JOIN s2_tokens s2
        ON s1.country = s2.country
        AND s1.token = s2.token;
""")

count_2b_s2 = con.execute("SELECT COUNT(*) FROM pairs_2b_s2").fetchone()[0]
print(f"Step 2b (S2) completed in {time.time() - start:.2f} seconds.")
print(f"Generated {count_2b_s2:,} candidate pairs from Source 2.")

Generating candidate pairs for Step 2b (Per-Token Explode - Source 2)...
Step 2b (S2) completed in 96.67 seconds.
Generated 175,269,873 candidate pairs from Source 2.


In [11]:
# Python Code Cell: Step 2b for Source 3

print("Generating candidate pairs for Step 2b (Per-Token Explode - Source 3)...")
start = time.time()

con.execute("""
    CREATE TABLE pairs_2b_s3 AS
    WITH s1_tokens AS (
        SELECT 
            entity_id AS s1_id,
            country,
            UNNEST(string_split(clean_name, ' ')) AS token
        FROM s1
        WHERE clean_name IS NOT NULL AND clean_name != ''
    ),
    s3_tokens AS (
        SELECT 
            entity_id AS s3_id,
            country,
            UNNEST(string_split(clean_name, ' ')) AS token
        FROM s3
        WHERE clean_name IS NOT NULL AND clean_name != ''
    ),
    -- Guardrail: Identify high-frequency tokens per country in S1 to drop
    valid_s1_tokens AS (
        SELECT 
            s1_id,
            country,
            token
        FROM (
            SELECT 
                s1_id,
                country,
                token,
                COUNT(*) OVER (PARTITION BY country, token) AS token_freq
            FROM s1_tokens
            WHERE LENGTH(token) >= 3
        )
        WHERE token_freq <= 300
    )
    SELECT DISTINCT
        s1.s1_id AS source1_entity_id,
        s3.s3_id AS candidate_entity_id,
        'S3' AS source,
        '2b_token_explode' AS matched_via
    FROM valid_s1_tokens s1
    INNER JOIN s3_tokens s3
        ON s1.country = s3.country
        AND s1.token = s3.token;
""")

count_2b_s3 = con.execute("SELECT COUNT(*) FROM pairs_2b_s3").fetchone()[0]
print(f"Step 2b (S3) completed in {time.time() - start:.2f} seconds.")
print(f"Generated {count_2b_s3:,} candidate pairs from Source 3.")

Generating candidate pairs for Step 2b (Per-Token Explode - Source 3)...
Step 2b (S3) completed in 131.63 seconds.
Generated 185,103,616 candidate pairs from Source 3.


### **2c. Tertiary Net: Address Channel Blocking**
For records where `clean_address` is available (non-null and non-empty), we block on matching standardized address strings within the same country. 

**Key Behavior:**
- Records with missing addresses (~3.3% in S2/S3) simply skip this net without causing errors.
- Those missing-address records are already fully covered by Nets 2a and 2b.
- This net catches entities located at the exact same physical address whose business names are completely different or misspelled.

In [12]:
# -------------------------------------------------------------------------
# STEP 2c: Address Channel Blocking for Source 2
# -------------------------------------------------------------------------
print("Generating candidate pairs for Step 2c (Address Channel - Source 2)...")
start = time.time()

con.execute("""
    CREATE TABLE pairs_2c_s2 AS
    SELECT DISTINCT
        s1.entity_id AS source1_entity_id,
        s2.entity_id AS candidate_entity_id,
        'S2' AS source,
        '2c_address_channel' AS matched_via
    FROM s1
    INNER JOIN s2
        ON s1.country = s2.country
       AND s1.clean_address = s2.clean_address
    WHERE s1.clean_address IS NOT NULL AND s1.clean_address != ''
      AND s2.clean_address IS NOT NULL AND s2.clean_address != '';
""")

count_2c_s2 = con.execute("SELECT COUNT(*) FROM pairs_2c_s2").fetchone()[0]
print(f"Step 2c (S2) completed in {time.time() - start:.2f} seconds.")
print(f"Generated {count_2c_s2:,} candidate pairs from Source 2.\n")

# -------------------------------------------------------------------------
# STEP 2c: Address Channel Blocking for Source 3
# -------------------------------------------------------------------------
print("Generating candidate pairs for Step 2c (Address Channel - Source 3)...")
start = time.time()

con.execute("""
    CREATE TABLE pairs_2c_s3 AS
    SELECT DISTINCT
        s1.entity_id AS source1_entity_id,
        s3.entity_id AS candidate_entity_id,
        'S3' AS source,
        '2c_address_channel' AS matched_via
    FROM s1
    INNER JOIN s3
        ON s1.country = s3.country
       AND s1.clean_address = s3.clean_address
    WHERE s1.clean_address IS NOT NULL AND s1.clean_address != ''
      AND s3.clean_address IS NOT NULL AND s3.clean_address != '';
""")

count_2c_s3 = con.execute("SELECT COUNT(*) FROM pairs_2c_s3").fetchone()[0]
print(f"Step 2c (S3) completed in {time.time() - start:.2f} seconds.")
print(f"Generated {count_2c_s3:,} candidate pairs from Source 3.")

Generating candidate pairs for Step 2c (Address Channel - Source 2)...
Step 2c (S2) completed in 2.68 seconds.
Generated 564,542 candidate pairs from Source 2.

Generating candidate pairs for Step 2c (Address Channel - Source 3)...
Step 2c (S3) completed in 2.04 seconds.
Generated 205,584 candidate pairs from Source 3.


## **Phase 2: The Multi-Net Blocking Strategy (Execution Summary)**

In this phase, we moved away from memory-heavy pandas and leveraged DuckDB out-of-core processing. Because comparing 2.2 million Source 1 records against 10.3 million S2/S3 records requires over 22 trillion direct comparisons, we built three independent blocking "nets." 

Every net was strictly constrained by a **Hard Country Partition** (`s1.country = s2.country`), instantly halving the search space and preventing impossible cross-border matches.

### **Net 2a: Sorted Token Blocking (The Word-Swap Catcher)**
*   **The Problem:** Exact string matching fails if words are out of order (*"moncada learning center"* vs *"learning center moncada"*).
*   **The Logic:** We split `clean_name` into an array of words, sorted them alphabetically, and rejoined them into a single string. 
*   **The Result:** Generated ~39.9M pairs for S2 and ~44.3M pairs for S3.

### **Net 2b: Per-Token Explode Blocking (The Missing-Word Catcher)**
*   **The Problem:** Sorted tokens fail if one source drops or adds a word (*"EFS Print Ventures"* vs *"EFS Print Ventures Group"*).
*   **The Logic:** We exploded the `clean_name` into individual words, generating a candidate pair if the businesses shared even a single word.
*   **The Guardrails:** To prevent a memory-crashing pair explosion from generic words, we strictly filtered out words shorter than 3 letters and words appearing more than 300 times in a single country.
*   **The Result:** Generated ~175.2M pairs for S2 and ~185.1M pairs for S3.

### **Net 2c: Address Channel Blocking (The Mangled Name Catcher)**
*   **The Problem:** Sometimes the business name is completely unrecognizable due to severe typos, but the physical location is an exact match.
*   **The Logic:** We matched strictly on the `clean_address`. We safely ignored the ~3.3% of records with missing addresses—they skip this net but are still protected by 2a and 2b.
*   **The Result:** Generated ~564k pairs for S2 and ~205k pairs for S3.


---


### **3. Union & Deduplication**
We generated roughly 445 million raw candidate pairs across our three blocking nets. Because we cast a wide net to maximize recall, millions of these pairs are identical duplicates (e.g., a true match was caught by both Net 2a and Net 2b). 

Here we combine all pairs using `UNION` (which automatically drops identical rows) and compress the `matched_via` tags so we know exactly which combination of rules successfully flagged each pair.


In [16]:
# Tune settings for maximum safety
con.execute("SET memory_limit='11GB'")
con.execute("SET threads=2")  # Lower threads further to eliminate buffer bloat
con.execute("SET preserve_insertion_order=false")

print("Building candidate_pairs table structure...")
# 1. Create the final table schema upfront
con.execute("""
    CREATE TABLE candidate_pairs (
        source1_entity_id VARCHAR,
        candidate_entity_id VARCHAR,
        source VARCHAR,
        matched_via VARCHAR
    );
""")

print("Appending Net 2a (Sorted Tokens) for S2...")
con.execute("""
    INSERT INTO candidate_pairs 
    SELECT source1_entity_id, candidate_entity_id, source, matched_via 
    FROM pairs_2a_s2;
""")

print("Appending Net 2b (Exploded Tokens) for S2...")
con.execute("""
    INSERT INTO candidate_pairs 
    SELECT source1_entity_id, candidate_entity_id, source, matched_via 
    FROM pairs_2b_s2;
""")

print("Appending Net 2c (Address Channel) for S2...")
con.execute("""
    INSERT INTO candidate_pairs 
    SELECT source1_entity_id, candidate_entity_id, source, matched_via 
    FROM pairs_2c_s2;
""")

print("Appending Net 2a (Sorted Tokens) for S3...")
con.execute("""
    INSERT INTO candidate_pairs 
    SELECT source1_entity_id, candidate_entity_id, source, matched_via 
    FROM pairs_2a_s3;
""")

print("Appending Net 2b (Exploded Tokens) for S3...")
con.execute("""
    INSERT INTO candidate_pairs 
    SELECT source1_entity_id, candidate_entity_id, source, matched_via 
    FROM pairs_2b_s3;
""")

print("Appending Net 2c (Address Channel) for S3...")
con.execute("""
    INSERT INTO candidate_pairs 
    SELECT source1_entity_id, candidate_entity_id, source, matched_via 
    FROM pairs_2c_s3;
""")

print("\n🎉 All nets successfully appended! Total raw rows:")
print(f"{con.execute('SELECT COUNT(*) FROM candidate_pairs').fetchone()[0]:,}")

Building candidate_pairs table structure...
Appending Net 2a (Sorted Tokens) for S2...
Appending Net 2b (Exploded Tokens) for S2...
Appending Net 2c (Address Channel) for S2...
Appending Net 2a (Sorted Tokens) for S3...
Appending Net 2b (Exploded Tokens) for S3...
Appending Net 2c (Address Channel) for S3...

🎉 All nets successfully appended! Total raw rows:
445,449,498
